Dans ce TP, nous considérons des trajets en vélo partagé (similaire au vélib) en Californie. Deux jeux de données sont fournis : l'un qui contient les stations de vélo, l'autre, les trajets à vélo. Les déplacements à vélo se font d'une station à l'autre.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab3TripApp") \
    .getOrCreate()

sc = spark.sparkContext

Charger le fichier station_data.csv dans un DataFrame station_df et le fichier trip_data.csv dans un DataFrame trip_df. Pour chaque Dataframe, il vous faudra demander une inférence des schémas et indiquer que la première ligne est un en-tête.

In [2]:
trip_df = spark.read.csv("../../data/trip_data.csv", header=True, inferSchema=True)
station_df = spark.read.option("header", "true").option("inferSchema", "true").csv("../../data/station_data.csv")

Afficher les schémas des 2 DataFrames.

In [3]:
print(trip_df.printSchema())
print(station_df.printSchema())

root
 |-- TripID: integer (nullable = true)
 |-- Duration: integer (nullable = true)
 |-- StartDate: string (nullable = true)
 |-- StartStation: string (nullable = true)
 |-- StartTerminal: integer (nullable = true)
 |-- EndDate: string (nullable = true)
 |-- EndStation: string (nullable = true)
 |-- EndTerminal: integer (nullable = true)
 |-- Bike#: integer (nullable = true)
 |-- SubscriberType: string (nullable = true)
 |-- ZipCode: string (nullable = true)

None
root
 |-- station_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dockcount: integer (nullable = true)
 |-- landmark: string (nullable = true)
 |-- installation: string (nullable = true)

None


In [4]:
station_df.toPandas()

,station_id,name,lat,long,dockcount,landmark,installation
0,2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013
1,3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013
2,4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013
3,5,Adobe on Almaden,37.331415,-121.893200,19,San Jose,8/5/2013
4,6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013
...,...,...,...,...,...,...,...
65,77,Market at Sansome,37.789625,-122.400811,27,San Francisco,8/25/2013
66,80,Santa Clara County Civic Center,37.352601,-121.905733,15,San Jose,12/31/2013
67,82,Broadway St at Battery St,37.798541,-122.400862,15,San Francisco,1/22/2014
68,83,Mezes Park,37.491269,-122.236234,15,Redwood City,2/20/2014


In [5]:
trip_df.toPandas()

,TripID,Duration,StartDate,StartStation,StartTerminal,EndDate,EndStation,EndTerminal,Bike#,SubscriberType,ZipCode
0,913460,765,8/31/2015 23:26,Harry Bridges Plaza (Ferry Building),50,8/31/2015 23:39,San Francisco Caltrain (Townsend at 4th),70,288,Subscriber,2139
1,913459,1036,8/31/2015 23:11,San Antonio Shopping Center,31,8/31/2015 23:28,Mountain View City Hall,27,35,Subscriber,95032
2,913455,307,8/31/2015 23:13,Post at Kearny,47,8/31/2015 23:18,2nd at South Park,64,468,Subscriber,94107
3,913454,409,8/31/2015 23:10,San Jose City Hall,10,8/31/2015 23:17,San Salvador at 1st,8,68,Subscriber,95113
4,913453,789,8/31/2015 23:09,Embarcadero at Folsom,51,8/31/2015 23:22,Embarcadero at Sansome,60,487,Customer,9069
...,...,...,...,...,...,...,...,...,...,...,...
354147,432951,619,9/1/2014 4:21,Powell Street BART,39,9/1/2014 4:32,Townsend at 7th,65,335,Subscriber,94118
354148,432950,6712,9/1/2014 3:16,Harry Bridges Plaza (Ferry Building),50,9/1/2014 5:08,San Francisco Caltrain (Townsend at 4th),70,259,Customer,44100
354149,432949,538,9/1/2014 0:05,South Van Ness at Market,66,9/1/2014 0:14,5th at Howard,57,466,Customer,32
354150,432948,568,9/1/2014 0:05,South Van Ness at Market,66,9/1/2014 0:15,5th at Howard,57,461,Customer,32


Créer une vue pour chaque DataFrame.

In [6]:
trip_df.createOrReplaceTempView("trip")
station_df.createOrReplaceTempView("station")

Trouver deux façons de calculer le nombre de trajets, l'une en appelant une méthode sur trip_df directement, l'autre en rédigeant une requête SQL de la vue correspondant au DataFrame tripData.

In [7]:
print(trip_df.count())
spark.sql(
    """
    SELECT COUNT(*) FROM trip
    """
).show()

354152
+--------+
|count(1)|
+--------+
|  354152|
+--------+



Ecrire une requête permettant de compter le nombre de trajets qui démarrent et se terminent à la même station.

In [8]:
spark.sql(
    """
    SELECT COUNT(*) FROM trip
    WHERE StartTerminal = EndTerminal
    """
).show()

+--------+
|count(1)|
+--------+
|   10276|
+--------+



In [9]:
trip_df.filter(trip_df["StartTerminal"] == trip_df["EndTerminal"]).toPandas()

,TripID,Duration,StartDate,StartStation,StartTerminal,EndDate,EndStation,EndTerminal,Bike#,SubscriberType,ZipCode
0,913212,73,8/31/2015 18:39,5th at Howard,57,8/31/2015 18:40,5th at Howard,57,456,Subscriber,94107
1,913211,7558,8/31/2015 18:38,Embarcadero at Sansome,60,8/31/2015 20:44,Embarcadero at Sansome,60,385,Customer,34
2,913210,7472,8/31/2015 18:38,Embarcadero at Sansome,60,8/31/2015 20:43,Embarcadero at Sansome,60,585,Customer,34
3,913198,353,8/31/2015 18:34,Townsend at 7th,65,8/31/2015 18:40,Townsend at 7th,65,370,Subscriber,94118
4,912838,631,8/31/2015 17:02,University and Emerson,35,8/31/2015 17:13,University and Emerson,35,253,Customer,6907
...,...,...,...,...,...,...,...,...,...,...,...
10271,432994,7503,9/1/2014 8:55,Powell at Post (Union Square),71,9/1/2014 11:00,Powell at Post (Union Square),71,400,Customer,nil
10272,432979,1290,9/1/2014 8:30,Davis at Jackson,42,9/1/2014 8:52,Davis at Jackson,42,462,Subscriber,94111
10273,432966,17396,9/1/2014 7:37,Mountain View City Hall,27,9/1/2014 12:27,Mountain View City Hall,27,140,Customer,94085
10274,432965,17297,9/1/2014 7:37,Mountain View City Hall,27,9/1/2014 12:25,Mountain View City Hall,27,57,Customer,94085


In [10]:
from pyspark.sql.functions import col
trip_df.filter(col("StartTerminal") == col("EndTerminal")).toPandas()

,TripID,Duration,StartDate,StartStation,StartTerminal,EndDate,EndStation,EndTerminal,Bike#,SubscriberType,ZipCode
0,913212,73,8/31/2015 18:39,5th at Howard,57,8/31/2015 18:40,5th at Howard,57,456,Subscriber,94107
1,913211,7558,8/31/2015 18:38,Embarcadero at Sansome,60,8/31/2015 20:44,Embarcadero at Sansome,60,385,Customer,34
2,913210,7472,8/31/2015 18:38,Embarcadero at Sansome,60,8/31/2015 20:43,Embarcadero at Sansome,60,585,Customer,34
3,913198,353,8/31/2015 18:34,Townsend at 7th,65,8/31/2015 18:40,Townsend at 7th,65,370,Subscriber,94118
4,912838,631,8/31/2015 17:02,University and Emerson,35,8/31/2015 17:13,University and Emerson,35,253,Customer,6907
...,...,...,...,...,...,...,...,...,...,...,...
10271,432994,7503,9/1/2014 8:55,Powell at Post (Union Square),71,9/1/2014 11:00,Powell at Post (Union Square),71,400,Customer,nil
10272,432979,1290,9/1/2014 8:30,Davis at Jackson,42,9/1/2014 8:52,Davis at Jackson,42,462,Subscriber,94111
10273,432966,17396,9/1/2014 7:37,Mountain View City Hall,27,9/1/2014 12:27,Mountain View City Hall,27,140,Customer,94085
10274,432965,17297,9/1/2014 7:37,Mountain View City Hall,27,9/1/2014 12:25,Mountain View City Hall,27,57,Customer,94085


In [11]:
trip_df.filter("StartTerminal = EndTerminal").toPandas()

,TripID,Duration,StartDate,StartStation,StartTerminal,EndDate,EndStation,EndTerminal,Bike#,SubscriberType,ZipCode
0,913212,73,8/31/2015 18:39,5th at Howard,57,8/31/2015 18:40,5th at Howard,57,456,Subscriber,94107
1,913211,7558,8/31/2015 18:38,Embarcadero at Sansome,60,8/31/2015 20:44,Embarcadero at Sansome,60,385,Customer,34
2,913210,7472,8/31/2015 18:38,Embarcadero at Sansome,60,8/31/2015 20:43,Embarcadero at Sansome,60,585,Customer,34
3,913198,353,8/31/2015 18:34,Townsend at 7th,65,8/31/2015 18:40,Townsend at 7th,65,370,Subscriber,94118
4,912838,631,8/31/2015 17:02,University and Emerson,35,8/31/2015 17:13,University and Emerson,35,253,Customer,6907
...,...,...,...,...,...,...,...,...,...,...,...
10271,432994,7503,9/1/2014 8:55,Powell at Post (Union Square),71,9/1/2014 11:00,Powell at Post (Union Square),71,400,Customer,nil
10272,432979,1290,9/1/2014 8:30,Davis at Jackson,42,9/1/2014 8:52,Davis at Jackson,42,462,Subscriber,94111
10273,432966,17396,9/1/2014 7:37,Mountain View City Hall,27,9/1/2014 12:27,Mountain View City Hall,27,140,Customer,94085
10274,432965,17297,9/1/2014 7:37,Mountain View City Hall,27,9/1/2014 12:25,Mountain View City Hall,27,57,Customer,94085


On souhaite désormais obtenir l’id des stations associées à ces trajets. Ecrire une requête renvoyant la liste des terminaux concernés ainsi que le nombre de trajets pour chacun de ces terminaux. Trier le résultat par ordre décroissant de nombre de trajets.
<br>Exemple de sortie :
<br>+--------+--------+
<br>|terminal|count(1)|
<br>+--------+--------+
<br>| 60| 850|
<br>| 50| 708|
<br>| 35| 348|
<br>| 76| 320|
<br>| 74| 307|
<br>(La station 60 est la plus concernée par ces trajets cycliques, avec 850 de ces trajets.)

In [12]:
spark.sql(
    """
    SELECT StartTerminal as Terminal, COUNT(*) as trip_count FROM trip
    WHERE StartTerminal = EndTerminal
    GROUP BY Terminal
    ORDER BY trip_count DESC
    """
).show()

+--------+----------+
|Terminal|trip_count|
+--------+----------+
|      60|       850|
|      50|       708|
|      35|       348|
|      76|       320|
|      74|       307|
|      39|       296|
|      61|       280|
|      67|       277|
|      71|       268|
|      70|       260|
|      28|       254|
|      48|       248|
|      54|       230|
|      69|       227|
|      42|       213|
|      73|       200|
|      57|       197|
|      64|       194|
|       3|       189|
|      72|       181|
+--------+----------+
only showing top 20 rows



Dans la requête précédente, nous avons oublié un élément qui nous importe. Nous souhaitons compléter le résultat en indiquant le nombre de docks (dockcount) des stations concernées.
<br>Exemple de sortie :
<br>+--------+---------+--------+
<br>|terminal|dockcount|count(1)|
<br>+--------+---------+--------+
<br>| 60| 15| 850|
<br>| 50| 23| 708|
<br>| 35| 11| 348|
<br>| 76| 19| 320|
<br>| 74| 23| 307|
<br>Mettre à jour la requête.

In [13]:
spark.sql(
    """
    SELECT StartTerminal as Terminal, COUNT(*) as trip_count, dockcount FROM trip
    JOIN station
    ON trip.StartTerminal = station.station_id
    WHERE StartTerminal = EndTerminal
    GROUP BY Terminal, dockcount
    ORDER BY trip_count DESC
    """
).show()

+--------+----------+---------+
|Terminal|trip_count|dockcount|
+--------+----------+---------+
|      60|       850|       15|
|      50|       708|       23|
|      35|       348|       11|
|      76|       320|       19|
|      74|       307|       23|
|      39|       296|       19|
|      61|       280|       27|
|      67|       277|       27|
|      71|       268|       19|
|      70|       260|       19|
|      28|       254|       23|
|      48|       248|       15|
|      54|       230|       15|
|      69|       227|       23|
|      42|       213|       15|
|      73|       200|       15|
|      57|       197|       15|
|      64|       194|       15|
|       3|       189|       15|
|      72|       181|       23|
+--------+----------+---------+
only showing top 20 rows



Rédiger les 2 requêtes précédentes avec le DSL de DataFrame.

In [14]:
from pyspark.sql.functions import col, desc

In [15]:
type(trip_df)

pyspark.sql.dataframe.DataFrame

In [16]:
res_df = trip_df.filter(
    col("StartTerminal") == col("EndTerminal")
).groupBy(
    col("StartTerminal")
).count(
).join(
    station_df,
    on=col("StartTerminal")==col("station_id"),
    how="left",
).select(
    "StartTerminal", "count", "dockcount",
).orderBy(
    desc("count")
)

Observer le plan d’exécution des requêtes.

In [23]:
res_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=1386]
   +- Sort [count#215L DESC NULLS LAST], true, 0
      +- Exchange rangepartitioning(count#215L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1384]
         +- Project [StartTerminal#21, count#215L, dockcount#60]
            +- BroadcastHashJoin [StartTerminal#21], [station_id#56], LeftOuter, BuildRight, false
               :- HashAggregate(keys=[StartTerminal#21], functions=[count(1)])
               :  +- Exchange hashpartitioning(StartTerminal#21, 200), ENSURE_REQUIREMENTS, [plan_id=1377]
               :     +- HashAggregate(keys=[StartTerminal#21], functions=[partial_count(1)])
               :        +- Project [StartTerminal#21]
               :           +- Filter ((isnotnull(StartTerminal#21) AND isnotnull(EndTerminal#24)) AND (StartTerminal#21 = EndTerminal#24))
               :              +- FileScan csv [StartTerminal#21,EndTerminal#24]

In [21]:
res_df = res_df.repartition(2)

In [22]:
res_df.write.parquet("../../data/output/titi.parquet")

Arrêter la session Spark.